# Equations 3 & 4 — Oil Supply Shock Instrumental Variables

Replication code for the 2SLS regression from:

> Albertazzi, U., 't Hooft, J., & ter Steege, L. (2025).  
> *The causal effect of inflation on financial stability: evidence from history.*  
> ECB Working Paper Series No. 3108.

---

## Methodology

A limitation of the matched pegged-base approach (Equation 1) is that monetary policy
is not the only omitted variable that could confound the inflation-crisis relationship.
To address this we instrument annual inflation with **oil supply shocks**
(Baumeister & Hamilton 2019): changes in the global supply of oil are plausibly
exogenous to domestic financial conditions, yet pass through to consumer prices.

**First stage (Equation 3):**

$$
\Delta\pi_{i,t} = \alpha_i + \beta_0 + \beta_1 \epsilon^{\text{OIL}}_t
  + \sum_{j=0}^{4} \beta_{2,j} \Delta R_{i,t-j}
  + \sum_{j=0}^{4} \beta_{3,j} \Delta y_{i,t-j} + u_{i,t}
$$

**Second stage (Equation 4 / Table 5):**

$$
\text{crisis}_{i,t} = \alpha_i + \beta_0
  + \beta_1 \widehat{\Delta\pi}_{i,t-1}
  + \sum_{j=0}^{4} \beta_{2,j} \Delta R_{i,t-j}
  + \sum_{j=0}^{4} \beta_{3,j} \Delta y_{i,t-j} + \epsilon_{i,t}
$$

Sample: 1975–2020 (constrained by oil shock data availability).

## 1. Imports

In [12]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from utils import melt, add_const, winsor, display, display_w_std_errors, run_first_stage
from data import load_jst, load_oil_shocks, build_oil_shocks_panel, build_base_panel

## 2. Data

See `data.py` for full source details and download instructions.

In [13]:
raw_df     = load_jst()
oil_shocks = load_oil_shocks()

def pivot(v): return raw_df.pivot(index='year', columns='country', values=v)

## 3. Variable Construction

In [14]:
cpi_df    = pivot('cpi')
gdp_df    = pivot('rgdpbarro')
stir_df   = pivot('stir')
crisis_df = pivot('crisisJST')

# Broadcast annual oil shocks to all countries (same value each year)
oil_shocks_df = build_oil_shocks_panel(cpi_df, oil_shocks)

## 4. Regression Panel

Builds the (country, year) panel with the dependent variable, endogenous regressor,
instrument, and controls. The effective sample is 1976–2020: oil shock data starts
in 1975, and instrumented inflation enters the second stage with a one-year lag.

In [15]:
panel_df, covariates = build_base_panel(
    raw_df, crisis_df, cpi_df, gdp_df, stir_df, oil_shocks_df)

## 5. First Stage — Equation 3

Regresses annual inflation change on the oil supply shock and controls.
A negative `oil_shock` coefficient confirms instrument relevance: an adverse
supply shock raises prices.

In [17]:
first_stage_fit, fitted_df = run_first_stage(panel_df, covariates)

# Oil shock coefficient — should be negative and significant
display(first_stage_fit).iloc[:2]

,results
const,-0.493***
oil_shock,-0.060***


## 6. Second Stage — Equation 4 (Table 5)

Fitted values are shifted one year and used as the instrumented inflation regressor.
The coefficient of interest is `delta_inflation_hat_1` ($\hat{\beta}_1$).

The paper reports $\hat{\beta}_1 = 0.046^{***}$ (0.017).

In [18]:
# Instrumented inflation, lagged one year
panel_df['delta_inflation_hat_1'] = melt(fitted_df.shift(1))

second_stage_vars = ['delta_inflation_hat_1'] + covariates
regression_df = panel_df[['crisis'] + second_stage_vars].dropna()

fit = PanelOLS(
    dependent      = regression_df['crisis'],
    exog           = add_const(regression_df[second_stage_vars]),
    entity_effects = True,
    time_effects   = False,
).fit(cov_type='clustered', cluster_entity=True)

display_w_std_errors(fit).iloc[:2]

,results_w_errors
const,0.051*** (0.01)
delta_inflation_hat_1,0.046*** (0.017)
